# 00 - LangChain 基础：Agent 开发的第一站

## 学习目标

- 理解 LangChain 的核心架构和设计理念
- 掌握 Chains、Prompts、Models、Memory 的基本用法
- 学习使用 LangChain 构建简单 Agent
- 了解 LangChain 的生态系统

---

## 1. LangChain 概述

### 1.1 什么是 LangChain？

**LangChain** 是一个用于开发 LLM 应用的 Python/JS 框架，由 Harrison Chase 于 2022 年创建。它提供了一套完整的工具和抽象，帮助开发者：

- **连接 LLM 与外部数据源**（文档、数据库、API）
- **构建复杂的工作流**（Chains、Agents）
- **管理对话状态**（Memory）
- **标准化开发流程**（统一的接口和组件）

### 1.2 LangChain 核心组件

```
┌─────────────────────────────────────────────────────────────┐
│                     LangChain 架构                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────────┐  ┌─────────────┐  ┌─────────────┐        │
│  │   Models    │  │   Prompts   │  │   Parsers   │        │
│  │             │  │             │  │             │        │
│  │ • ChatModels│  │ • Templates │  │ • StrOutput │        │
│  │ • LLMs      │  │ • Examples  │  │ • JsonOutput│        │
│  │ • Embeddings│  │ • Selectors │  │ • Pydantic  │        │
│  └──────┬──────┘  └──────┬──────┘  └──────┬──────┘        │
│         │                │                │               │
│         └────────────────┼────────────────┘               │
│                          │                                 │
│                   ┌──────┴──────┐                          │
│                   │    Chains   │                          │
│                   │             │                          │
│                   │ • LLMChain  │                          │
│                   │ • Router    │                          │
│                   │ • Sequential│                          │
│                   └──────┬──────┘                          │
│                          │                                 │
│         ┌────────────────┼────────────────┐               │
│         │                │                │               │
│    ┌────┴────┐     ┌────┴────┐     ┌────┴────┐          │
│    │  Memory │     │  Agents │     │ Callback│          │
│    │         │     │         │     │         │          │
│    │• Buffer │     │• ReAct  │     │• Logging│          │
│    │• Summary│     │• Plan   │     │• Tracing│          │
│    │• Vector │     │• Tool   │     │• Custom │          │
│    └─────────┘     └─────────┘     └─────────┘          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 1.3 LangChain 版本说明

- **LangChain v0.1** (2024)：稳定版，核心 API 确定
- **LangChain v0.2** (2024)：改进的架构，更好的可组合性
- **LangChain v0.3** (2025)：增强的 Agent 支持，性能优化

**注意**：LangChain 正在向 **LangGraph** 演进，用于更复杂的状态管理。

---

## 2. 环境准备

### 2.1 安装依赖



In [ ]:
# 安装 LangChain 核心库
# !pip install langchain langchain-core langchain-community -q

# 安装模型相关库（根据使用的模型选择）
# !pip install langchain-openai -q  # OpenAI
# !pip install langchain-anthropic -q  # Claude
# !pip install langchain-google-genai -q  # Google Gemini

# 安装其他依赖
# !pip install python-dotenv -q

import os
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

print("环境准备完成")
print(f"LangChain 版本: 待安装后查看")



---
## 0. 模型准备：加载 Qwen2.5-7B-Instruct

本 Notebook 使用 **ModelScope** 加载本地 Qwen 模型，替代在线 API 调用。

- **推荐模型**：`Qwen/Qwen2.5-7B-Instruct`（约 15GB 显存）
- **低显存备选**：`Qwen/Qwen2.5-3B-Instruct`（约 6GB 显存）
- 自动检测 GPU / CPU，优先使用 GPU 加速

> 如果没有安装 modelscope 或显存不足，可以使用下方代码中的 **MockLLM 备选方案**。



In [ ]:
# ============================================================
# 安装依赖（首次运行时取消注释）
# ============================================================
# !pip install modelscope torch transformers -q

import torch

# ============================================================
# GPU / CPU 自动检测
# ============================================================
if torch.cuda.is_available():
    DEVICE = "cuda"
    GPU_NAME = torch.cuda.get_device_name(0)
    print(f"检测到 GPU: {GPU_NAME}")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("检测到 Apple Silicon GPU (MPS)")
else:
    DEVICE = "cpu"
    print("未检测到 GPU，将使用 CPU（速度较慢）")

# ============================================================
# QwenLLM 封装类
# ============================================================
from modelscope import AutoModelForCausalLM, AutoTokenizer

class QwenLLM:
    """基于 ModelScope 的 Qwen 模型封装"""

    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct", device=None):
        # 自动检测设备
        if device is None:
            device = "cuda" if torch.cuda.is_available() else (
                "mps" if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
                else "cpu"
            )
        self.device = device
        print(f"正在加载模型 {model_name}，设备: {device} ...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype="auto", device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.messages = []
        print("模型加载完成！")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """对话接口"""
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})
        return response

    def reset(self):
        """清空对话历史"""
        self.messages = []

# ============================================================
# 初始化模型
# ============================================================
# 低显存环境可切换为: Qwen/Qwen2.5-3B-Instruct
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

try:
    llm = QwenLLM(model_name=MODEL_NAME)
    USE_REAL_MODEL = True
    print("\n使用真实 Qwen 模型")
except Exception as e:
    print(f"\n模型加载失败: {e}")
    print("将使用 MockLLM 备选方案")
    USE_REAL_MODEL = False

print(f"\n当前设备: {DEVICE}")
print(f"使用真实模型: {USE_REAL_MODEL}")



In [ ]:
# ============================================================
# 无模型时的备选方案：MockLLM
# （仅在上方模型加载失败时使用）
# ============================================================

if not USE_REAL_MODEL:
    class MockLLM:
        """模拟 LLM，用于无模型环境下的教学演示"""

        def __init__(self, model_name="mock-qwen"):
            self.model_name = model_name
            self.messages = []

        def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
            """模拟对话回复"""
            if '你好' in user_message:
                response = "你好！我是 AI 助手，有什么可以帮助你的吗？"
            elif '天气' in user_message:
                response = "我无法获取实时天气信息，建议您查看天气应用。"
            elif 'LangChain' in user_message:
                response = "LangChain 是一个用于开发 LLM 应用的框架，提供了 Chains、Prompts、Memory 等核心组件。"
            elif 'Agent' in user_message:
                response = "Agent 是一种能够自主决策并调用工具的智能体，通常使用 ReAct 模式工作。"
            else:
                response = f"我收到了您的消息：'{user_message[:50]}'。这是一个模拟回复。"

            self.messages.append({"role": "user", "content": user_message})
            self.messages.append({"role": "assistant", "content": response})
            return response

        def reset(self):
            self.messages = []

    llm = MockLLM(model_name="mock-qwen")
    print("已启用 MockLLM 备选方案")

# 测试模型调用
response = llm.chat("你好，请介绍一下自己")
print(f"模型回复: {response}")



### 2.2 配置 API 密钥

在使用真实模型前，需要配置 API 密钥。建议将密钥保存在 `.env` 文件中：

```bash
# .env 文件
OPENAI_API_KEY=your_openai_key
ANTHROPIC_API_KEY=your_anthropic_key
GOOGLE_API_KEY=your_google_key

# 国产模型
ZHIPUAI_API_KEY=your_zhipu_key
DASHSCOPE_API_KEY=your_dashscope_key
```

---

## 3. 核心组件详解

### 3.1 Models（模型）

LangChain 支持多种 LLM 和 ChatModel：



In [ ]:
# ============================================================
# 使用 QwenLLM 替代 MockChatModel
# ============================================================

# --- 无模型时的备选方案：MockChatModel（已注释）---
# class MockChatModel:
#     """模拟 ChatModel"""
#     def __init__(self, model_name="mock-model"):
#         self.model_name = model_name
#     def invoke(self, messages):
#         user_msg = None
#         for msg in messages:
#             if msg.get('role') == 'user' or msg.get('type') == 'human':
#                 user_msg = msg
#         content = user_msg.get('content', '') if user_msg else ''
#         if '你好' in content:
#             response = "你好！我是 AI 助手，有什么可以帮助你的吗？"
#         elif '天气' in content:
#             response = "我无法获取实时天气信息，建议您查看天气应用。"
#         else:
#             response = f"我收到了您的消息：'{content[:50]}...'。这是一个模拟回复。"
#         return {'content': response, 'role': 'assistant', 'model': self.model_name}
#     def predict(self, text):
#         return self.invoke([{'role': 'user', 'content': text}])['content']

# --- 使用真实 Qwen 模型 ---
# llm 已在上方加载（QwenLLM 实例）

# 测试对话
response = llm.chat("你好，请介绍一下自己")
print(f"模型回复: {response}")

# 测试带 system_prompt 的调用
response2 = llm.chat(
    "什么是 ReAct 模式？",
    system_prompt="你是一个 AI Agent 专家，请用中文简洁回答。"
)
print(f"\n专家回复: {response2}")



**实际使用 LangChain 的代码**：

```python
# OpenAI
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4", temperature=0.7)

# Anthropic Claude
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-3-sonnet-20240229")

# Google Gemini
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-pro")

# 智谱 AI
from langchain_community.chat_models import ChatZhipuAI
llm = ChatZhipuAI(model="glm-4")

# 通义千问
from langchain_community.chat_models.tongyi import ChatTongyi
llm = ChatTongyi(model_name="qwen-max")
```

### 3.2 Prompts（提示词）

LangChain 提供了强大的提示词管理功能：



In [ ]:
# 提示词模板示例

class MockPromptTemplate:
    """模拟 PromptTemplate"""

    def __init__(self, template, input_variables):
        self.template = template
        self.input_variables = input_variables

    def format(self, **kwargs):
        """格式化模板"""
        result = self.template
        for key, value in kwargs.items():
            result = result.replace(f'{{{key}}}', str(value))
        return result

    def invoke(self, inputs):
        """调用模板"""
        return self.format(**inputs)

# 创建提示词模板
template = """
你是一个专业的{role}。

请根据以下上下文回答问题：
{context}

问题：{question}

请用{language}回答。
""",

prompt = MockPromptTemplate(
    template=template,
    input_variables=["role", "context", "question", "language"]
)

# 使用模板
formatted_prompt = prompt.format(
    role="技术顾问",
    context="我们正在开发一个 AI Agent 系统",
    question="什么是 ReAct 模式？",
    language="中文"
)

print("格式化后的提示词：")
print("=" * 50)
print(formatted_prompt)
print("=" * 50)



**实际使用 LangChain 的代码**：

```python
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 基础模板
prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个专业的{role}。"),
    ("human", "请回答：{question}")
])

# 带历史消息的模板
prompt_with_history = ChatPromptTemplate.from_messages([
    ("system", "你是一个 helpful 的 AI 助手。"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# Few-shot 模板
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "2+2", "output": "4"},
    {"input": "3*3", "output": "9"}
]

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=ChatPromptTemplate.from_messages([
        ("human", "{input}"),
        ("ai", "{output}")
    ]),
    examples=examples
)
```

### 3.3 Chains（链）

Chain 是 LangChain 的核心概念，将多个组件串联起来：



In [ ]:
# Chain 概念演示（使用 QwenLLM）

class SimpleChain:
    """简化版 Chain 实现，使用 QwenLLM"""

    def __init__(self, prompt, llm):
        self.prompt = prompt
        self.llm = llm

    def invoke(self, inputs):
        """执行 Chain"""
        # Step 1: 格式化提示词
        formatted_prompt = self.prompt.invoke(inputs)
        print(f"[Chain] 格式化提示词:")
        print(f"{formatted_prompt[:100]}...")

        # Step 2: 调用 QwenLLM
        response = self.llm.chat(formatted_prompt)
        print(f"[Chain] 模型回复:")
        print(f"{response[:100]}...")

        return {
            'input': inputs,
            'output': response
        }

# 创建 Chain
qa_prompt = MockPromptTemplate(
    template="""
你是一个 AI 助手。请回答用户的问题。

问题：{question}
""",
    input_variables=["question"]
)

chain = SimpleChain(prompt=qa_prompt, llm=llm)

# 执行 Chain
result = chain.invoke({"question": "什么是 LangChain？"})
print(f"\n最终结果: {result['output']}")



**实际使用 LangChain 的代码**：

```python
from langchain_core.runnables import RunnableSequence
from langchain_core.output_parsers import StrOutputParser

# 构建 Chain
chain = (
    prompt
    | llm
    | StrOutputParser()
)

# 执行
result = chain.invoke({"question": "什么是 LangChain？"})

# 更复杂的 Chain
from operator import itemgetter

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question")
    }
    | prompt
    | llm
    | StrOutputParser()
)
```

### 3.4 Memory（记忆）

LangChain 提供了多种记忆实现：



In [ ]:
# Memory 概念演示

class MockConversationBufferMemory:
    """模拟 ConversationBufferMemory"""

    def __init__(self, memory_key="history", return_messages=True):
        self.memory_key = memory_key
        self.return_messages = return_messages
        self.messages = []

    def add_user_message(self, message):
        """添加用户消息"""
        self.messages.append({"role": "user", "content": message})

    def add_ai_message(self, message):
        """添加 AI 消息"""
        self.messages.append({"role": "assistant", "content": message})

    def load_memory_variables(self, inputs):
        """加载记忆变量"""
        if self.return_messages:
            return {self.memory_key: self.messages}
        else:
            # 返回文本格式
            history = "\n".join([
                f"{msg['role']}: {msg['content']}"
                for msg in self.messages
            ])
            return {self.memory_key: history}

    def save_context(self, inputs, outputs):
        """保存上下文"""
        if 'input' in inputs:
            self.add_user_message(inputs['input'])
        if 'output' in outputs:
            self.add_ai_message(outputs['output'])

    def clear(self):
        """清空记忆"""
        self.messages = []

# 创建记忆实例
memory = MockConversationBufferMemory()

# 模拟对话
memory.add_user_message("你好，我叫张三")
memory.add_ai_message("你好张三！很高兴认识你。有什么可以帮助你的吗？")
memory.add_user_message("我想学习 Python")
memory.add_ai_message("Python 是一门很好的编程语言！我可以推荐一些学习资源。")

# 查看记忆
print("对话历史：")
print("=" * 50)
vars = memory.load_memory_variables({})
for msg in vars['history']:
    print(f"[{msg['role']}] {msg['content']}")
print("=" * 50)



**实际使用 LangChain 的代码**：

```python
from langchain.memory import (
    ConversationBufferMemory,
    ConversationBufferWindowMemory,
    ConversationSummaryMemory,
    VectorStoreRetrieverMemory
)

# 基础缓冲记忆
memory = ConversationBufferMemory(
    memory_key="history",
    return_messages=True
)

# 窗口记忆（只保留最近 k 轮）
memory = ConversationBufferWindowMemory(
    k=5,  # 保留最近 5 轮
    memory_key="history"
)

# 摘要记忆（自动摘要历史）
memory = ConversationSummaryMemory(
    llm=llm,  # 用于生成摘要
    memory_key="history"
)

# 向量检索记忆
from langchain_community.vectorstores import Chroma
memory = VectorStoreRetrieverMemory(
    retriever=vectorstore.as_retriever()
)
```

---

## 4. 使用 LangChain 构建 Agent

### 4.1 ReAct Agent



In [ ]:
# 模拟 LangChain Agent 概念

class MockTool:
    """模拟 LangChain Tool"""

    def __init__(self, name, func, description):
        self.name = name
        self.func = func
        self.description = description

    def run(self, query):
        return self.func(query)

class MockAgentExecutor:
    """模拟 AgentExecutor"""

    def __init__(self, agent, tools, verbose=True):
        self.agent = agent
        self.tools = {tool.name: tool for tool in tools}
        self.verbose = verbose
        self.max_iterations = 5

    def invoke(self, inputs):
        """执行 Agent"""
        query = inputs['input']

        print(f"\n{'='*60}")
        print(f"Agent 执行: {query}")
        print(f"{'='*60}")

        # 模拟 ReAct 循环
        for i in range(self.max_iterations):
            print(f"\n第 {i+1} 步")

            # 使用 QwenLLM 进行思考
            thought = self._think_with_llm(query)
            print(f"Thought: {thought}")

            # 模拟行动选择
            action = self._choose_action(query)

            if action == 'finish':
                answer = self._generate_answer_with_llm(query)
                print(f"\n最终答案: {answer}")
                return {'input': query, 'output': answer}

            print(f"Action: {action}")

            # 执行工具
            if action in self.tools:
                observation = self.tools[action].run(query)
                print(f"Observation: {observation}")

        return {'input': query, 'output': '达到最大迭代次数'}

    def _think_with_llm(self, query):
        """使用 QwenLLM 进行思考"""
        try:
            prompt = f"分析用户查询 '{query}'，判断需要什么工具或是否可以直接回答。简短回答。"
            return llm.chat(prompt, max_new_tokens=100)
        except Exception:
            if '天气' in query:
                return '用户询问天气，我需要使用天气工具'
            elif '计算' in query:
                return '这是一个计算问题，我需要使用计算工具'
            else:
                return '我可以直接回答这个问题'

    def _choose_action(self, query):
        if '天气' in query:
            return 'weather_tool'
        elif '计算' in query:
            return 'calculator_tool'
        else:
            return 'finish'

    def _generate_answer_with_llm(self, query):
        """使用 QwenLLM 生成最终答案"""
        try:
            return llm.chat(f"请回答用户的问题：{query}", max_new_tokens=256)
        except Exception:
            return f'关于 "{query}" 的答案是：这是一个模拟回答。'

# 定义工具
def get_weather(query):
    return "北京：晴天，25°C"

def calculate(query):
    return "计算结果：42"

tools = [
    MockTool("weather_tool", get_weather, "获取天气信息"),
    MockTool("calculator_tool", calculate, "执行计算")
]

# 创建 Agent
agent_executor = MockAgentExecutor(
    agent="react",
    tools=tools
)

print("Agent 初始化完成")



In [ ]:
# 测试 Agent
result = agent_executor.invoke({"input": "今天北京的天气怎么样？"})
print(f"\n最终结果: {result['output']}")



**实际使用 LangChain 的代码**：

```python
from langchain.agents import create_react_agent, AgentExecutor
from langchain_core.tools import Tool
from langchain import hub

# 获取 ReAct prompt
prompt = hub.pull("hwchase17/react")

# 创建 Agent
agent = create_react_agent(llm, tools, prompt)

# 创建执行器
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=5
)

# 执行
result = agent_executor.invoke({"input": "今天北京的天气怎么样？"})

# 使用更简单的预构建 Agent
from langchain.agents import create_tool_calling_agent

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)
```

---

## 5. LangChain 生态系统

### 5.1 相关项目

| 项目 | 描述 | 用途 |
|------|------|------|
| **LangChain** | 核心框架 | LLM 应用开发 |
| **LangGraph** | 状态图工作流 | 复杂 Agent 编排 |
| **LangServe** | 部署服务 | 将 Chain 部署为 API |
| **LangSmith** | 观测平台 | 调试、监控、评估 |
| **LangChain Templates** | 模板库 | 快速启动项目 |

### 5.2 与其他框架的关系

```
LangChain (基础框架)
    ├── LangGraph (复杂工作流)
    │       └── 状态机、循环、条件分支
    ├── LangServe (部署)
    │       └── REST API、Stream
    └── LangSmith (观测)
            └── Trace、Eval、Monitor
```

---

## 6. 小结

### 核心要点

1. **LangChain** 是 LLM 应用开发的基础框架，提供统一的组件接口
2. **核心组件**：Models、Prompts、Chains、Memory、Agents
3. **Chain** 是串联组件的核心机制，支持复杂工作流
4. **Agent** 通过 ReAct 等模式实现自主决策和工具调用
5. **生态系统**：LangGraph（工作流）、LangServe（部署）、LangSmith（观测）

### 下一步

- [01_langgraph_workflows.ipynb](01_langgraph_workflows.ipynb) - 学习 LangGraph 状态图工作流
- [02_autogen_multi_agent.ipynb](02_autogen_multi_agent.ipynb) - 探索 AutoGen 多 Agent 框架

---

## 参考资源

- [LangChain 官方文档](https://python.langchain.com/)
- [LangChain Cookbook](https://github.com/langchain-ai/langchain/blob/master/cookbook/)
- [LangChain 概念指南](https://python.langchain.com/docs/concepts/)

